# 🔐 Clortho — Project Documentation

**A fully local, encrypted password manager built in Python**

---

> *No cloud. No accounts. No telemetry. Your credentials stay on your machine, encrypted with AES-256, and never leave without your explicit permission.*

This notebook documents the architecture, security model, features, and design decisions behind Clortho — covering the CLI core, the Flask web UI, and the Firefox browser extension.

---


## Table of Contents

1. [Project Overview](#1-project-overview)
2. [Security Architecture](#2-security-architecture)
3. [Component Breakdown](#3-component-breakdown)
4. [Encryption Deep Dive](#4-encryption-deep-dive)
5. [Network Firewall](#5-network-firewall)
6. [CLI Features](#6-cli-features)
7. [Web UI Features](#7-web-ui-features)
8. [Browser Extension](#8-browser-extension)
9. [Data Import & Export](#9-data-import--export)
10. [Password Generator](#10-password-generator)
11. [File & Permission Model](#11-file--permission-model)
12. [Threat Model](#12-threat-model)
13. [Dependency Audit](#13-dependency-audit)
14. [Quick-Start Guide](#14-quick-start-guide)
15. [Bazzite Startup Script](#15-bazzite-startup-script)


---
## 1. Project Overview

Clortho is a **three-layer** password management system:

| Layer | File | Purpose |
|---|---|---|
| **Core / CLI** | `clortho.py` | Encryption engine + interactive terminal shell |
| **Web UI** | `clortho_web.py` | Local Flask server with browser-based GUI |
| **Extension** | `clortho_extension/` | Firefox extension for autofill on login pages |

All three share the **same encrypted vault file** (`~/.clortho/vault.vk`). There is no synchronization needed — the CLI and Web UI read/write the same file directly, and the extension talks to the Web UI over localhost.

### Design philosophy

- **Offline-first**: Nothing requires an internet connection except the optional webpage-reader feature, which only connects when explicitly invoked
- **Transparent**: All code is readable Python and vanilla JavaScript — no compiled binaries, no obfuscated logic
- **Minimal permissions**: The extension only requests `activeTab` and access to `127.0.0.1:7777`
- **No recovery by design**: If you forget your master password, the vault cannot be opened. This is a feature, not a bug — it means there's no backdoor.


---
## 2. Security Architecture

### Security layers at a glance


In [ ]:
# Security properties summary — run this cell to display the model
properties = {
    "Cipher":              "AES-256-GCM  (via Python cryptography / Fernet)",
    "Key derivation":      "PBKDF2-SHA256, 480,000 iterations  (OWASP 2023)",
    "Salt":                "32 bytes, cryptographically random, unique per vault",
    "Vault format":        "Single encrypted binary file  (vault.vk)",
    "File permissions":    "chmod 600  (owner read/write only)",
    "Atomic writes":       "Temp file + os.replace()  (no partial-write corruption)",
    "Password in memory":  "Cleared with del pw after key derivation",
    "Wrong-password limit":"3 attempts then process exits",
    "Min master password": "12 characters enforced",
    "Network":             "Blocked at socket level except 127.0.0.1 and explicit webpage command",
    "Plaintext on disk":   "Never  (except explicit user export, which warns and sets chmod 600)",
    "Telemetry":           "None",
    "Update checks":       "None",
    "Cloud sync":          "None",
}

col_w = max(len(k) for k in properties) + 2
print(f"{'Property':<{col_w}}  Value")
print("─" * 80)
for k, v in properties.items():
    print(f"  {k:<{col_w}}{v}")


---
## 3. Component Breakdown

### File structure

```
clortho/
│
├── clortho.py          # Core: crypto engine + CLI shell
├── clortho_web.py      # Web UI: Flask server (127.0.0.1 only)
├── clortho_start.sh    # Bazzite/Flatpak launcher script
│
└── web_extensions/         # Firefox extension (Manifest V2)
    ├── manifest.json       # Extension metadata + permissions
    ├── background.js       # Service worker: vault API relay + match scoring
    ├── content.js          # Injected into pages: form detection + autofill UI
    ├── popup.html          # Toolbar popup: status + fill + password generator
    ├── popup.js            # Popup logic
    └── icons/
        ├── icon48.png
        └── icon96.png

~/.clortho/             # Default vault directory (created automatically)
    ├── vault.vk            # Encrypted vault  (AES-256-GCM)
    └── .vk_salt            # 32-byte random salt

~/.local/share/clortho/extension/   # Deployed extension (for about:debugging)
```

### How the components communicate

```
  Firefox Extension
       │
       │  Bearer token auth  (Authorization: Bearer <token>)
       │  fetch() to 127.0.0.1:7777
       ▼
  clortho_web.py  ◄──────────►  vault.vk  (on disk)
  (Flask, localhost)               (encrypted)
       ▲
       │  shared file
       │
  clortho.py
  (CLI, same vault)
```

### Authentication flow

Session cookies use `SameSite=Lax`, which browsers block on cross-origin fetches from
`moz-extension://` origins. Clortho solves this with a Bearer token relay:

1. On successful unlock, the server generates `_api_token = secrets.token_hex(32)`
2. The token is embedded in the vault page as `<meta name="clortho-api-token" content="...">`
3. The content script reads it and sends `STORE_TOKEN` to the background service worker
4. The background stores it in `browser.storage.local` and uses it for all subsequent
   `Authorization: Bearer <token>` headers
5. The server validates the token in the `require_unlock` decorator alongside the session cookie

The web server holds the decrypted vault **in memory** after unlock. The CLI reads and
decrypts on each run. The extension never holds any vault data — it only relays requests
through the web server session.


---
## 4. Encryption Deep Dive

### Key derivation with PBKDF2-SHA256


In [ ]:
# Demonstrate the key derivation process (with a dummy password — never use this)
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
import base64, secrets, time

ITERATIONS = 480_000

salt = secrets.token_bytes(32)
password = "DemoPassword_DoNotUse"

start = time.perf_counter()
kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=salt, iterations=ITERATIONS)
key = base64.urlsafe_b64encode(kdf.derive(password.encode()))
elapsed = time.perf_counter() - start

print(f"Iterations : {ITERATIONS:,}")
print(f"Salt       : {salt.hex()[:32]}…  ({len(salt)} bytes)")
print(f"Derived key: {key[:32]}…  (256-bit, base64url)")
print(f"Time taken : {elapsed*1000:.0f} ms")
print()
print("This ~300-500ms delay is intentional — it makes brute-force attacks")
print("~480,000x harder than a single SHA-256 hash.")


### Fernet (AES-256-GCM) encryption

Fernet is a high-level symmetric encryption scheme from the `cryptography` library. It wraps:
- **AES-256-CBC** for confidentiality  
- **HMAC-SHA256** for authentication (detects tampering)
- A **timestamp** embedded in the token (used for optional TTL)

The key insight is that Fernet is **authenticated** — if anyone modifies even a single byte of the vault file, decryption will fail with `InvalidToken` rather than silently returning garbage data.


In [ ]:
# Demonstrate encrypt/decrypt round-trip
from cryptography.fernet import Fernet, InvalidToken
import json

# In real use, this key comes from PBKDF2 above
key = Fernet.generate_key()
f = Fernet(key)

vault_data = {
    "entries": [
        {"site": "GitHub", "username": "demo@example.com", "password": "s3cr3t!"}
    ],
    "meta": {"version": "1.1"}
}

# Encrypt
plaintext  = json.dumps(vault_data).encode()
ciphertext = f.encrypt(plaintext)
print(f"Plaintext  : {len(plaintext)} bytes")
print(f"Ciphertext : {len(ciphertext)} bytes")
print(f"Ciphertext : {ciphertext[:60]}…")

# Decrypt
recovered = json.loads(f.decrypt(ciphertext).decode())
assert recovered == vault_data
print("\nDecryption: OK ✓")

# Tamper detection
tampered = bytearray(ciphertext)
tampered[40] ^= 0xFF  # flip a bit
try:
    f.decrypt(bytes(tampered))
except InvalidToken:
    print("Tamper detection: InvalidToken raised ✓  (vault corruption detected)")


---
## 5. Network Firewall

One of Clortho's more unusual features is a **runtime socket-level firewall** that prevents any accidental or malicious network connections from within the Python process.

### How it works


In [ ]:
# Illustrate the socket guard pattern used in clortho.py
import socket

_LOOPBACK = {"127.0.0.1", "::1", "localhost", "0.0.0.0"}
_network_allowed = False  # global gate — only True during webpage fetch

_orig_connect     = socket.socket.connect
_orig_getaddrinfo = socket.getaddrinfo

def _guarded_connect(self, address):
    host = address[0] if isinstance(address, (tuple, list)) else str(address)
    if host in _LOOPBACK or _network_allowed:
        return _orig_connect(self, address)
    raise PermissionError(f"Blocked outbound connection to {address}")

def _guarded_getaddrinfo(*args, **kwargs):
    host = str(args[0]) if args else ""
    if host in _LOOPBACK or _network_allowed:
        return _orig_getaddrinfo(*args, **kwargs)
    raise PermissionError(f"Blocked DNS lookup for '{host}'")

# Install the guard
socket.socket.connect = _guarded_connect
socket.getaddrinfo    = _guarded_getaddrinfo

# Test it
print("Testing socket guard...")

# Loopback must work (Flask server needs this)
result = socket.getaddrinfo("127.0.0.1", 7777, socket.AF_INET, socket.SOCK_STREAM)
print(f"  127.0.0.1 resolution: allowed ✓  ({result[0][4]})")

# External connections must be blocked
try:
    socket.getaddrinfo("google.com", 443)
    print("  External DNS: NOT blocked  ✗")
except PermissionError as e:
    print(f"  External DNS: blocked ✓")

try:
    s = socket.socket()
    s.connect(("8.8.8.8", 53))
    print("  External TCP: NOT blocked  ✗")
except PermissionError:
    print(f"  External TCP: blocked ✓")
finally:
    # Restore for this notebook
    socket.socket.connect = _orig_connect
    socket.getaddrinfo    = _orig_getaddrinfo

print("\nGuard operates at the Python socket layer — catches any library,")
print("import, or code that tries to make an outbound connection.")


### The `webpage` command — the only escape hatch

When the user runs the `webpage` command, the guard is temporarily lifted for a single HTTP request:

```python
_network_allowed = True          # open the gate
try:
    resp = requests.get(url, ...)
finally:
    _network_allowed = False     # always re-lock, even on exception
```

The user is shown the target URL and must confirm before any connection is made. `requests` and `BeautifulSoup` are **lazy imports** — they're only loaded into memory if this command is actually used.


---
## 6. CLI Features

The CLI is the foundation — it works completely without Flask or a browser.

### Commands

| Command | Aliases | Description |
|---|---|---|
| `list` | `ls`, `l` | Show all entries (passwords masked by default) |
| `add` | `new`, `a` | Add a new credential entry |
| `search` | `find`, `s`, `f` | Search by site, username, URL, category, or notes |
| `edit` | `update`, `e` | Edit an existing entry by ID |
| `delete` | `rm`, `del` | Remove an entry by ID (with confirmation) |
| `import` | `imp`, `i` | Import from CSV or Excel |
| `webpage` | `web`, `w` | Fetch a URL and add its credentials |
| `generate` | `gen`, `g` | Generate a strong random password |
| `export` | — | Export vault to plaintext CSV (double-confirmed) |
| `help` | `?`, `h` | Show command reference |
| `quit` | `q`, `exit` | Exit and lock vault |

### Entry data model


In [ ]:
# Every vault entry stores these fields
import json

example_entry = {
    "id":       "a3f8c1d2e9b7",     # 64-bit random hex, generated at creation
    "site":     "GitHub",
    "username": "matt@example.com",
    "password": "••••••••••••",     # stored encrypted in vault
    "url":      "https://github.com",
    "notes":    "Work account — 2FA enabled",
    "category": "Development",
    "created":  "2024-11-15T09:23:41.112233",
    "modified": "2024-11-15T09:23:41.112233",
}

print("Entry fields:")
for k, v in example_entry.items():
    print(f"  {k:<12} {v}")

print()
print("All entries live in vault.data['entries'] as a list.")
print("The entire list is serialized to JSON, then AES-256 encrypted as one blob.")
print("There is no per-field encryption — the whole vault is atomic.")


---
## 7. Web UI Features

`clortho_web.py` launches a Flask server that binds **exclusively to `127.0.0.1`** — it is never accessible from other machines on your network.

### REST API

| Method | Route | Description |
|---|---|---|
| `GET` | `/api/entries` | List all entries (requires unlocked session) |
| `POST` | `/api/entries` | Create a new entry |
| `PUT` | `/api/entries/<id>` | Update an existing entry |
| `DELETE` | `/api/entries/<id>` | Delete an entry |
| `POST` | `/api/import` | Upload and import a CSV/Excel file |
| `GET` | `/api/generate` | Generate a random password |
| `POST` | `/api/lock` | Lock the vault and clear session |
| `GET/POST` | `/unlock` | Vault unlock page |

### Session security

- Session cookie is `HttpOnly` and `SameSite=Lax`
- Session key is `secrets.token_bytes(32)` — regenerated every server start
- `debug=False` is hardcoded — cannot be accidentally enabled
- Uploaded import files go to a system temp directory and are `os.unlink()`-ed immediately after parsing

### UI features

- **Category sidebar** — auto-populated from your data
- **Slide-in detail panel** — click any entry to expand
- **Password masking** — show/hide toggle per field
- **Copy to clipboard** — auto-clears after 30 seconds
- **Drag-and-drop import** — drop a CSV or Excel file onto the import zone
- **Inline password generator** — length slider from 12–64 characters
- **Keyboard shortcuts** — `Ctrl/Cmd+K` to focus search, `Esc` to close panel


In [ ]:
# Demonstrate the CORS policy — only extension and localhost origins allowed
ALLOWED_ORIGINS = {"http://127.0.0.1:7777", "http://localhost:7777"}

def cors_policy(origin: str) -> bool:
    """Returns True if the origin should receive CORS headers."""
    return (
        origin in ALLOWED_ORIGINS
        or origin.startswith("moz-extension://")    # Firefox extension
        or origin.startswith("chrome-extension://") # Chromium extension
    )

test_origins = [
    ("http://127.0.0.1:7777",         True,  "Local web UI"),
    ("moz-extension://abc123def456",  True,  "Firefox extension"),
    ("chrome-extension://xyz789",     True,  "Chromium extension"),
    ("https://evil.example.com",      False, "External origin"),
    ("https://google.com",            False, "External origin"),
    ("*",                             False, "Wildcard (never)"),
    ("null",                          False, "Null origin"),
]

print(f"{'Origin':<42} {'Allowed':<8} {'Reason'}")
print("─" * 70)
for origin, expected, reason in test_origins:
    result = cors_policy(origin)
    status = "✓ YES" if result else "✗ NO "
    flag   = "" if result == expected else "  ← UNEXPECTED"
    print(f"  {origin:<40} {status:<8} {reason}{flag}")


---
## 8. Browser Extension

The extension is a standard Firefox WebExtension (Manifest V2) with three JavaScript
contexts communicating via `browser.runtime.sendMessage`.

### Architecture

```
┌─────────────────────────────────────────────────────────┐
│  Browser Tab (any website)                              │
│  ┌──────────────────────────────────────────────────┐   │
│  │  content.js                                      │   │
│  │  • Watches for password field focus (focusin)    │   │
│  │  • Renders floating autofill prompt (shadow DOM) │   │
│  │  • simulateFill() triggers React/Vue/Angular     │   │
│  │  • On 127.0.0.1: reads clortho-api-token meta tag     │   │
│  │    and relays it to background via STORE_TOKEN   │   │
│  └──────────────┬───────────────────────────────────┘   │
└─────────────────┼───────────────────────────────────────┘
                  │ browser.runtime.sendMessage
                  ▼
┌─────────────────────────────────────────────────────────┐
│  background.js (Service Worker)                         │
│  • STORE_TOKEN  → persist Bearer token to storage       │
│  • GET_MATCHES  → fetch /api/entries, score by hostname │
│  • GET_PASSWORD → fetch /api/entries, return password   │
│  • CHECK_STATUS → probe server health                   │
└──────────────────────┬──────────────────────────────────┘
                       │ fetch() + Authorization: Bearer <token>
                       ▼
                 clortho_web.py
```

### Popup features

The toolbar popup (`popup.html` / `popup.js`) provides:

| Feature | Details |
|---|---|
| **Open Vault** | Header button always visible — opens `http://127.0.0.1:7777` |
| **Status indicator** | Green (unlocked) / yellow (locked) / red (server offline) |
| **Credential matches** | Lists entries matching the current tab's hostname |
| **Fill button** | Clicks fill both username and password fields via the content script |
| **Password generator** | Length slider (8–64), A–Z / a–z / 0–9 / !@# toggles, copy button |

The generator uses `crypto.getRandomValues()` (browser CSPRNG) and excludes ambiguous
characters (0, 1, l, I, O) by default.

### Permissions requested

```json
"permissions": [
    "activeTab",               // Read URL of current tab only
    "storage",                 // Persist Bearer token across background restarts
    "http://127.0.0.1:7777/*"  // Talk to local vault server
]
```

Notably absent: `tabs`, `history`, `bookmarks`, `cookies`, `webRequest`, `<all_urls>` in permissions.

### CORS configuration

The server's `add_cors` after-request hook grants cross-origin access only to
`moz-extension://` and `chrome-extension://` origins. The allowed headers include
`Authorization` so the Bearer token preflight succeeds:

```python
response.headers["Access-Control-Allow-Headers"] = "Content-Type, Authorization"
```

### Installation

**Temporary (per session)** — no tools required:  
`about:debugging` → **Load Temporary Add-on** → select `web_extensions/manifest.json`

**Permanent (signed via AMO)**:
```bash
npm install -g web-ext
web-ext sign --api-key=<jwt_issuer> --api-secret=<jwt_secret> \
             --source-dir=./web_extensions --channel=unlisted
```
Get API credentials at `https://addons.mozilla.org/en-US/developers/addon/api/key/`

### Entry matching algorithm


In [ ]:
# The hostname matching algorithm used in background.js (ported to Python)

def score_entry(entry: dict, hostname: str) -> int:
    """Score how well a vault entry matches the current page hostname."""
    from urllib.parse import urlparse

    def normalize(s: str) -> str:
        try:
            if not s.startswith("http"):
                s = "https://" + s
            return urlparse(s).hostname.lstrip("www.") or ""
        except Exception:
            return s.lower().lstrip("www.")

    host       = hostname.lstrip("www.")
    entry_host = normalize(entry.get("url") or entry.get("site") or "")

    if not entry_host:
        return 0
    if entry_host == host:
        return 100                          # Exact match
    if host.endswith("." + entry_host) or entry_host.endswith("." + host):
        return 80                           # Subdomain match
    site_name = (entry.get("site") or "").lower()
    host_base = host.split(".")[0]
    if site_name == host_base or site_name in host_base or host_base in site_name:
        return 50                           # Fuzzy site-name match
    return 0

# Test cases
entries = [
    {"site": "GitHub",   "url": "https://github.com",       "username": "dev@example.com"},
    {"site": "Gmail",    "url": "https://mail.google.com",   "username": "matt@gmail.com"},
    {"site": "AWS",      "url": "https://console.aws.amazon.com", "username": "aws@example.com"},
    {"site": "Netflix",  "url": "",                          "username": "matt@gmail.com"},
]

test_pages = ["github.com", "mail.google.com", "auth.github.com", "netflix.com", "amazon.com"]

print(f"{'Page hostname':<28}", end="")
for e in entries:
    print(f"  {e['site']:<10}", end="")
print()
print("─" * 72)

for host in test_pages:
    print(f"  {host:<26}", end="")
    for e in entries:
        score = score_entry(e, host)
        marker = f"{score:>3}" if score > 0 else "  —"
        print(f"  {marker:<12}", end="")
    print()

print()
print("Scores: 100=exact  80=subdomain  50=fuzzy name  —=no match")


---
## 9. Data Import & Export

### Import

Clortho can import from `.csv`, `.xlsx`, `.xls`, and `.xlsm` files. The column detector uses flexible alias matching, so exports from major password managers work out of the box.

| Field | Accepted column names |
|---|---|
| Site | `site`, `name`, `website`, `service`, `title`, `account` |
| Username | `username`, `user`, `email`, `login`, `email address` |
| Password | `password`, `pass`, `passwd`, `pwd` |
| URL | `url`, `link`, `web address` *(optional)* |
| Notes | `notes`, `note`, `comment` *(optional)* |
| Category | `category`, `group`, `folder` *(optional)* |

**Compatible with direct exports from:** Chrome, Firefox, Edge, Safari, 1Password, Bitwarden, LastPass, Dashlane, KeePass.


In [ ]:
# Demonstrate the column alias detection
import pandas as pd
import io

# Simulate a Chrome browser export
chrome_export = """name,url,username,password
GitHub,https://github.com,dev@example.com,gh_secret123
Gmail,https://gmail.com,matt@gmail.com,gm_secret456
"""

# Simulate a LastPass export
lastpass_export = """url,username,password,totp,extra,name,grouping,fav
https://github.com,dev@example.com,gh_secret123,,,GitHub,Work,0
https://gmail.com,matt@gmail.com,gm_secret456,,,Gmail,Personal,0
"""

COL_ALIASES = {
    "site":     ["site", "name", "website", "service", "title", "account"],
    "username": ["username", "user", "email", "login", "email address"],
    "password": ["password", "pass", "passwd", "pwd"],
    "url":      ["url", "link", "web address", "address"],
}

def detect_columns(df):
    cols = [c.strip().lower() for c in df.columns]
    detected = {}
    for field, aliases in COL_ALIASES.items():
        for alias in aliases:
            if alias in cols:
                detected[field] = alias
                break
    return detected

for name, csv_data in [("Chrome export", chrome_export), ("LastPass export", lastpass_export)]:
    df = pd.read_csv(io.StringIO(csv_data))
    df.columns = [c.strip().lower() for c in df.columns]
    detected = detect_columns(df)
    print(f"{name}:")
    print(f"  Raw columns : {list(df.columns)}")
    print(f"  Detected    : {detected}")
    required_ok = all(k in detected for k in ("site", "username", "password"))
    print(f"  Importable  : {'✓ Yes' if required_ok else '✗ No'}\n")


### Export

Export is deliberately inconvenient — by design:

1. Two confirmation prompts (interactive + `Confirm.ask`)
2. Warning message displayed before and after
3. Output file immediately set to `chmod 600`
4. The export is intended for migration, not regular use

The vault should never need to be exported in normal operation.


---
## 10. Password Generator

Clortho uses Python's `secrets` module — which draws from the OS CSPRNG (`/dev/urandom` on Linux/macOS, `BCryptGenRandom` on Windows) — rather than the `random` module.


In [ ]:
import secrets
import string
from collections import Counter

def generate_password(length: int = 20) -> str:
    """Cryptographically secure password meeting all complexity requirements."""
    alphabet = string.ascii_letters + string.digits + "!@#$%^&*()-_=+[]"
    # Rejection sampling: regenerate until all character classes are present
    while True:
        pw = "".join(secrets.choice(alphabet) for _ in range(length))
        if (any(c.isupper() for c in pw)
                and any(c.islower() for c in pw)
                and any(c.isdigit() for c in pw)
                and any(c in "!@#$%^&*()-_=+[]" for c in pw)):
            return pw

# Generate a sample and check entropy
import math

alphabet_size = len(string.ascii_letters + string.digits + "!@#$%^&*()-_=+[]")
pw_length = 20
entropy_bits = pw_length * math.log2(alphabet_size)

print(f"Alphabet size  : {alphabet_size} characters")
print(f"Password length: {pw_length}")
print(f"Entropy        : {entropy_bits:.1f} bits")
print(f"Combinations   : {alphabet_size**pw_length:.2e}")
print()

# Generate 5 examples
print("Sample passwords:")
for _ in range(5):
    pw = generate_password(20)
    classes = sum([
        any(c.isupper() for c in pw),
        any(c.islower() for c in pw),
        any(c.isdigit() for c in pw),
        any(c in "!@#$%^&*()-_=+[]" for c in pw),
    ])
    print(f"  {pw}   ({classes}/4 classes)")


---
## 11. File & Permission Model

### Vault directory layout

```
~/.clortho/          ← Created with os.makedirs(exist_ok=True)
  vault.vk               ← chmod 600 — the encrypted vault
  .vk_salt               ← chmod 600 — 32-byte random salt
```

Both files are set to `0o600` immediately after writing. On POSIX systems this means only the owning user can read or write them — no group access, no world access.

### Atomic write pattern

To prevent vault corruption from a crash mid-write:

```python
def _save(self):
    ciphertext = encrypt_vault(self.data, self._key)
    tmp_path   = self.vault_path.with_suffix(".tmp")
    tmp_path.write_bytes(ciphertext)   # write to .tmp first
    tmp_path.chmod(0o600)
    tmp_path.replace(self.vault_path)  # atomic rename on POSIX
```

`os.replace()` (called by `Path.replace()`) is atomic on POSIX — the vault file is either the old version or the new version, never a partial state.

### What's never written to disk

- Master password (cleared with `del pw` after key derivation)
- Derived key (held in `vault._key`, never serialized)
- Individual passwords (only exist in the encrypted vault blob)
- Uploaded import files (saved to system temp, unlinked immediately after parse)


---
## 12. Threat Model

### What Clortho protects against

| Threat | Protection |
|---|---|
| Vault file stolen from disk | AES-256-GCM — unreadable without master password |
| Brute-force master password | 480,000-iteration PBKDF2 — ~400ms per attempt |
| Malicious Python import phoning home | Socket-level firewall blocks all external connections |
| Extension exfiltrating vault to remote server | Extension only has permission to `127.0.0.1:7777` |
| CORS attack from malicious webpage | Only `moz-extension://` and `localhost` origins allowed |
| Vault corruption from partial write | Atomic temp-file-then-rename pattern |
| Accidental plaintext export | Double-confirmation + chmod 600 on output |
| Weak master password | 12-character minimum enforced |
| Password brute-force via unlock UI | 3-attempt limit then process exits |

### What Clortho does NOT protect against

| Threat | Notes |
|---|---|
| Compromised OS / root access | An attacker with root can read process memory. No password manager protects against this. |
| Malicious browser extensions | A different extension with `<all_urls>` could intercept keystrokes. This is a browser-level risk. |
| Screen recording / keylogger | Physical security and OS hygiene are outside scope. |
| Forgotten master password | By design. No recovery mechanism exists. |
| Side-channel timing attacks | Not specifically hardened against; considered out of scope for local tool. |

### Extension trust boundary

The extension communicates with `clortho_web.py` over localhost. The trust model is:
- The **extension** is trusted code (you installed it)
- The **web server** validates session cookies on every request
- The **vault** is only decrypted in the server process memory
- Passwords flow: `vault.vk` → server RAM → background.js RAM → content.js fill → cleared

No password is ever stored in `localStorage`, `sessionStorage`, extension storage, or anywhere persistent.


---
## 13. Dependency Audit


In [ ]:
# Audit all dependencies — what they're used for and whether they touch the network

dependencies = [
    {
        "package":  "cryptography",
        "version":  ">=41.0",
        "used_for": "PBKDF2-SHA256 key derivation + Fernet (AES-256-GCM) encryption",
        "network":  False,
        "notes":    "Core security primitive. C-extension backed. Well-audited.",
    },
    {
        "package":  "pandas",
        "version":  ">=2.0",
        "used_for": "CSV and Excel file parsing for import",
        "network":  False,
        "notes":    "Only used during import. read_csv/read_excel are local operations.",
    },
    {
        "package":  "openpyxl",
        "version":  ">=3.1",
        "used_for": "Excel (.xlsx) file reading, called by pandas",
        "network":  False,
        "notes":    "Pure Python. Only loads local files.",
    },
    {
        "package":  "rich",
        "version":  ">=13.0",
        "used_for": "Terminal formatting, tables, prompts in CLI",
        "network":  False,
        "notes":    "Checked: no telemetry, no update checks, no network calls.",
    },
    {
        "package":  "flask",
        "version":  ">=3.0",
        "used_for": "Local web server for the GUI",
        "network":  False,
        "notes":    "Binds to 127.0.0.1 only. debug=False hardcoded.",
    },
    {
        "package":  "requests",
        "version":  ">=2.31",
        "used_for": "HTTP fetch for the webpage command only",
        "network":  "Conditional",
        "notes":    "Lazy import — only loaded if 'webpage' command is used.",
    },
    {
        "package":  "beautifulsoup4",
        "version":  ">=4.12",
        "used_for": "HTML parsing for the webpage command only",
        "network":  False,
        "notes":    "Pure parser — no network activity itself.",
    },
]

print(f"  {'Package':<16} {'Network?':<12} {'Used for'}")
print("  " + "─" * 78)
for d in dependencies:
    net = "⚠ Conditional" if d["network"] == "Conditional" else ("✗ Never" if not d["network"] else "YES")
    print(f"  {d['package']:<16} {net:<14} {d['used_for']}")

print()
print("All network access from dependencies is either:")
print("  • Impossible (no network code in the library for this use)")
print("  • Blocked by the socket guard")
print("  • Conditional on explicit user action (webpage command)")


---
## 14. Quick-Start Guide

### Prerequisites

```bash
pip install cryptography pandas openpyxl beautifulsoup4 requests rich flask
```

Python 3.10+ required.

---

### Step 1 — First run (creates your vault)

```bash
python clortho.py
```

You'll be prompted to choose a master password (minimum 12 characters).
There is no recovery — choose something memorable and strong.

---

### Step 2 — Using the CLI

```
clortho> add                    # Add a new entry
clortho> list                   # Show all entries
clortho> search github          # Find entries matching "github"
clortho> import ~/passwords.csv # Import from CSV or Excel
clortho> generate               # Generate a strong password
clortho> webpage                # Read a login page URL
clortho> quit                   # Lock and exit
```

---

### Step 3 — Launch the web UI

```bash
python clortho_web.py
# Opens http://127.0.0.1:7777
```

Enter your master password in the browser to unlock. The web UI and CLI share the same vault file.

---

### Step 4 — Install the Firefox extension

The extension source lives in `web_extensions/`. Deploy it to a permanent location:

```bash
mkdir -p ~/.local/share/clortho/extension
cp -r web_extensions/. ~/.local/share/clortho/extension/
```

#### Load into Firefox (temporary — required each session)

1. Open Firefox and navigate to `about:debugging`
2. Click **This Firefox** in the left sidebar
3. Click **Load Temporary Add-on...**
4. Navigate to `~/.local/share/clortho/extension/`
5. Select `manifest.json` and click Open
6. The 🔐 Clortho icon appears in the toolbar

> **Note:** Firefox removes temporary extensions on restart. See Step 5 to make it permanent.

---

### Step 5 — Make the extension permanent (signed via AMO)

Firefox Release requires extensions to be signed. The easiest path for a private extension
is an **unlisted** AMO submission — the extension is not publicly visible, just signed.

```bash
# Install the signing tool (one time)
npm install -g web-ext

# Sign the extension (get API credentials from addons.mozilla.org/developers/addon/api/key/)
web-ext sign \
  --api-key=<your_jwt_issuer> \
  --api-secret=<your_jwt_secret> \
  --source-dir=./web_extensions \
  --channel=unlisted

# Install the signed .xpi — it persists across restarts
# Firefox → about:addons → gear icon → Install Add-on From File → select the .xpi
```

---

### Step 6 — Autofill in practice

1. Start the server and unlock the vault at `http://127.0.0.1:7777/unlock`  
   (The extension reads the session token from the page automatically after unlock)
2. Navigate to any site you have a saved entry for
3. Click into a password field — the Clortho prompt appears
4. Click an entry to fill username and password

Click the 🔐 toolbar icon at any time to:
- See matching credentials for the current page
- Generate a new password
- Open the Clortho web UI directly

---

### Backup your vault

```bash
cp -r ~/.clortho ~/backups/clortho_$(date +%Y%m%d)/
```

Both `vault.vk` and `.vk_salt` are required to restore. Keep them together.

---

*Clortho is local-only. Read the source — there are no surprises.*


---
## 15. Bazzite Startup Script

On Bazzite Linux with Flatpak Firefox, unsigned extensions can't be permanently installed without developer tooling. The practical solution is a single script — `clortho_start.sh` — that does everything in one command each session:

1. **Starts the Clortho web server** (`clortho_web.py`)
2. **Rebuilds the extension `.xpi`** automatically if any source files changed
3. **Stages the `.xpi`** into your Firefox profile's extensions folder
4. **Launches Firefox** with `about:debugging` pre-opened so loading the extension is one click
5. **Tails the server log** so you see output and can unlock the vault
6. **Cleans up** the server process on `Ctrl+C`

### Installation

```bash
# Copy the script to your clortho folder (same place as clortho_web.py)
cp clortho_start.sh /path/to/your/clortho/
chmod +x /path/to/your/clortho/clortho_start.sh
```

### Usage

```bash
# Normal launch — starts server + Firefox
./clortho_start.sh

# Server only, no browser
./clortho_start.sh --no-browser

# Custom vault directory
./clortho_start.sh --vault ~/Documents/myvault

# Custom port
./clortho_start.sh --port=8888
```

### What happens each session

```
1. Run ./clortho_start.sh
2. Firefox opens with two tabs:
     - about:debugging  (for loading the extension)
     - http://127.0.0.1:7777/unlock  (to unlock your vault)
3. In about:debugging → click "Load Temporary Add-on..."
     → select the path shown in the terminal output
4. Enter your master password in the unlock tab
5. The 🔐 icon appears — you're ready
```

The loading step takes about 3 clicks and 10 seconds. `Ctrl+C` in the terminal stops the vault server and exits cleanly.

### Making it even faster — desktop shortcut

Create a `.desktop` file so you can launch everything from your app menu or taskbar:

```bash
# Adjust SCRIPT_DIR to wherever you saved clortho_start.sh
SCRIPT_DIR="/run/media/matt/Data Store/Unclassified/Claude/clortho-project/"

cat > ~/.local/share/applications/clortho.desktop << EOF
[Desktop Entry]
Name=Clortho
Comment=Launch Clortho vault server and Firefox
Exec=bash -c 'cd "${SCRIPT_DIR}" && ./clortho_start.sh'
Icon=dialog-password
Terminal=true
Type=Application
Categories=Utility;Security;
EOF

# Make it executable
chmod +x ~/.local/share/applications/clortho.desktop

# Refresh app menu
update-desktop-database ~/.local/share/applications/ 2>/dev/null || true
```

Clortho will now appear in your application menu. Clicking it opens a terminal running the startup script.


In [ ]:
# Display the full startup script inline
script = open('clortho_start.sh').read() if __import__('os').path.exists('clortho_start.sh') else open('/run/media/matt/Data Store/Unclassified/Claude/clortho-project/clortho_start.sh').read()

# Print with line numbers
for i, line in enumerate(script.splitlines(), 1):
    print(f"{i:>3}  {line}")


In [ ]:
# Generate a ready-to-use desktop shortcut for your actual path
import os

# Detect the script directory relative to where this notebook is running
script_dir = os.path.dirname(os.path.abspath('clortho_start.sh')) if os.path.exists('clortho_start.sh') else '~/clortho'

desktop_entry = f"""[Desktop Entry]
Name=Clortho
Comment=Launch Clortho vault server and Firefox
Exec=bash -c 'cd \"{script_dir}\" && ./clortho_start.sh'
Icon=dialog-password
Terminal=true
Type=Application
Categories=Utility;Security;
"""

desktop_path = os.path.expanduser('~/.local/share/applications/clortho.desktop')

print("Desktop file contents:")
print("─" * 50)
print(desktop_entry)
print("─" * 50)
print(f"\nTo install, run:")
print(f"  cat > {desktop_path} << 'EOF'")
print(desktop_entry.strip())
print("EOF")
print(f"  chmod +x {desktop_path}")
print(f"  update-desktop-database ~/.local/share/applications/")
